# **LIGHTWEIGHT 4D DIGITAL TWIN**

In [0]:
# =========================================================
# AEROFLUX - LIGHTWEIGHT 4D DIGITAL TWIN
# =========================================================
#
# SOURCE:
# /Volumes/aeroflux/gold_layer/gold_4d_trajectory
#
# 4D:
#   X = longitude
#   Y = latitude
#   Z = altitude
#   T = event_time
#
# DESIGN:
#   - one state per aircraft per minute
#   - one dynamic Scatter3D trace
#   - no historical trajectory line traces
#   - time slider + play/pause
#
# =========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import plotly.graph_objects as go


# =========================================================
# 1. PATH
# =========================================================

PATH = (
    "/Volumes/aeroflux/gold_layer/gold_4d_trajectory"
)


# =========================================================
# 2. READ GOLD 4D
# =========================================================

df = (
    spark.read
    .format("delta")
    .load(PATH)
)


# =========================================================
# 3. FILTER USABLE STATE
# =========================================================

df = (
    df
    .filter(
        F.col("icao24").isNotNull()
        &
        F.col("event_time").isNotNull()
        &
        F.col("latitude").isNotNull()
        &
        F.col("longitude").isNotNull()
        &
        F.col("baro_altitude_m").isNotNull()
    )
)


# =========================================================
# 4. CREATE 1-MINUTE TIME FRAME
# =========================================================

df = df.withColumn(
    "time_frame",
    F.date_trunc(
        "minute",
        F.col("event_time")
    )
)


# =========================================================
# 5. KEEP ONE LATEST OBSERVATION PER AIRCRAFT / MINUTE
# =========================================================

frame_window = (
    Window
    .partitionBy(
        "icao24",
        "time_frame"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("observation_id").desc()
    )
)


df = (
    df
    .withColumn(
        "row_number",
        F.row_number().over(
            frame_window
        )
    )
    .filter(
        F.col("row_number") == 1
    )
    .drop(
        "row_number"
    )
)


# =========================================================
# 6. SELECT ONLY WHAT THE VISUALIZATION NEEDS
# =========================================================

visual_df = (
    df
    .select(
        "icao24",
        "event_time",
        "time_frame",

        "latitude",
        "longitude",

        "baro_altitude_m",

        "velocity_mps",
        "true_track",

        "h3_res_8",

        "flight_session_id",
        "trajectory_segment_id"
    )
)


# =========================================================
# 7. SMALL PANDAS DATASET FOR VISUALIZATION
# =========================================================

pdf = visual_df.toPandas()


# =========================================================
# 8. NORMALIZE TIME
# =========================================================

pdf["event_time"] = pd.to_datetime(
    pdf["event_time"]
)

pdf["time_frame"] = pd.to_datetime(
    pdf["time_frame"]
)


pdf = (
    pdf
    .sort_values(
        [
            "time_frame",
            "icao24",
            "event_time"
        ]
    )
    .reset_index(drop=True)
)


# =========================================================
# 9. CREATE FRAME INDEX
# =========================================================
#
# Using integer frame IDs makes the Plotly animation
# simpler and lighter.
# =========================================================

unique_times = (
    pdf["time_frame"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

time_to_index = {
    timestamp: index
    for index, timestamp
    in enumerate(unique_times)
}


pdf["frame_id"] = pdf[
    "time_frame"
].map(
    time_to_index
)


# =========================================================
# 10. INITIAL FRAME
# =========================================================

initial_frame_id = int(
    pdf["frame_id"].min()
)

initial_df = (
    pdf[
        pdf["frame_id"]
        ==
        initial_frame_id
    ]
    .copy()
)


# =========================================================
# 11. HOVER DATA
# =========================================================

def build_customdata(data):

    if data.empty:
        return []

    return list(
        zip(
            data["icao24"].astype(str),

            data["event_time"].dt.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            data["velocity_mps"]
            .round(2),

            data["true_track"]
            .round(1),

            data["h3_res_8"].astype(str),

            data[
                "flight_session_id"
            ].astype(str),

            data[
                "trajectory_segment_id"
            ].astype(str)
        )
    )


initial_customdata = build_customdata(
    initial_df
)


# =========================================================
# 12. SINGLE DYNAMIC AIRCRAFT TRACE
# =========================================================

aircraft_trace = go.Scatter3d(

    x=initial_df[
        "longitude"
    ],

    y=initial_df[
        "latitude"
    ],

    z=(
        initial_df[
            "baro_altitude_m"
        ]
        / 1000.0
    ),

    mode="markers",

    marker=dict(
        size=6,
        opacity=0.85
    ),

    customdata=initial_customdata,

    hovertemplate=(
        "<b>Aircraft:</b> %{customdata[0]}<br>"
        "<b>Longitude:</b> %{x:.4f}<br>"
        "<b>Latitude:</b> %{y:.4f}<br>"
        "<b>Altitude:</b> %{z:.2f} km<br>"
        "<b>Velocity:</b> %{customdata[2]:.2f} m/s<br>"
        "<b>Track:</b> %{customdata[3]:.1f}°<br>"
        "<b>H3 R8:</b> %{customdata[4]}<br>"
        "<b>Event:</b> %{customdata[1]}<br>"
        "<extra></extra>"
    ),

    name="Aircraft"
)


# =========================================================
# 13. BUILD LIGHTWEIGHT ANIMATION FRAMES
# =========================================================

frames = []


for frame_id in sorted(
    pdf["frame_id"].unique()
):

    current = (
        pdf[
            pdf["frame_id"]
            ==
            frame_id
        ]
        .copy()
    )

    customdata = build_customdata(
        current
    )

    frames.append(
        go.Frame(

            name=str(
                int(frame_id)
            ),

            data=[
                go.Scatter3d(

                    x=current[
                        "longitude"
                    ],

                    y=current[
                        "latitude"
                    ],

                    z=(
                        current[
                            "baro_altitude_m"
                        ]
                        / 1000.0
                    ),

                    mode="markers",

                    marker=dict(
                        size=6,
                        opacity=0.85
                    ),

                    customdata=customdata,

                    hovertemplate=(
                        "<b>Aircraft:</b> %{customdata[0]}<br>"
                        "<b>Longitude:</b> %{x:.4f}<br>"
                        "<b>Latitude:</b> %{y:.4f}<br>"
                        "<b>Altitude:</b> %{z:.2f} km<br>"
                        "<b>Velocity:</b> %{customdata[2]:.2f} m/s<br>"
                        "<b>Track:</b> %{customdata[3]:.1f}°<br>"
                        "<b>H3 R8:</b> %{customdata[4]}<br>"
                        "<b>Event:</b> %{customdata[1]}<br>"
                        "<extra></extra>"
                    )
                )
            ],

            traces=[0]
        )
    )


# =========================================================
# 14. TIME SLIDER
# =========================================================

slider_steps = []


for index, timestamp in enumerate(
    unique_times
):

    label = pd.Timestamp(
        timestamp
    ).strftime(
        "%Y-%m-%d %H:%M"
    )

    slider_steps.append(
        dict(

            label=label,

            method="animate",

            args=[
                [str(index)],

                {
                    "mode": "immediate",

                    "frame": {
                        "duration": 400,
                        "redraw": True
                    },

                    "transition": {
                        "duration": 100
                    }
                }
            ]
        )
    )


# =========================================================
# 15. BUILD FIGURE
# =========================================================

fig = go.Figure(

    data=[
        aircraft_trace
    ],

    frames=frames
)


# =========================================================
# 16. LAYOUT
# =========================================================

fig.update_layout(

    title=(
        "AEROFLUX — 4D Airspace Digital Twin"
    ),

    height=750,

    scene=dict(

        xaxis_title="Longitude",

        yaxis_title="Latitude",

        zaxis_title="Altitude (km)"
    ),

    # -----------------------------------------------------
    # PLAY / PAUSE
    # -----------------------------------------------------

    updatemenus=[
        dict(

            type="buttons",

            showactive=False,

            x=0.05,

            y=1.08,

            buttons=[

                dict(

                    label="▶ Play",

                    method="animate",

                    args=[
                        None,

                        {
                            "frame": {
                                "duration": 500,
                                "redraw": True
                            },

                            "transition": {
                                "duration": 100
                            },

                            "fromcurrent": True,

                            "mode": "immediate"
                        }
                    ]
                ),

                dict(

                    label="⏸ Pause",

                    method="animate",

                    args=[
                        [None],

                        {
                            "frame": {
                                "duration": 0,
                                "redraw": False
                            },

                            "transition": {
                                "duration": 0
                            },

                            "mode": "immediate"
                        }
                    ]
                )
            ]
        )
    ],

    # -----------------------------------------------------
    # TIME SLIDER
    # -----------------------------------------------------

    sliders=[
        dict(

            active=0,

            currentvalue=dict(
                prefix="Simulation time: "
            ),

            pad=dict(
                t=45
            ),

            steps=slider_steps
        )
    ],

    margin=dict(
        l=0,
        r=0,
        b=0,
        t=75
    )
)


# =========================================================
# 17. DISPLAY
# =========================================================

fig.show()

<!doctype html>

# **XGBOOST**

In [0]:
# =========================================================
# AEROFLUX - FINAL XGBOOST CONGESTION FORECASTING
# =========================================================

# SOURCE:
#   /Volumes/aeroflux/gold_layer/gold_h3_traffic
#
# OUTPUT TABLES:
#   aeroflux.gold_layer.xgb_congestion_forecast
#   aeroflux.gold_layer.xgb_congestion_latest
#   aeroflux.gold_layer.xgb_congestion_metrics
#   aeroflux.gold_layer.xgb_congestion_feature_importance
#
# MODEL FILES:
#   /Volumes/aeroflux/gold_layer/gold_h3_congestion/xgb_models
#
# Horizons:
#   5 min
#   15 min
#   30 min
#
# =========================================================


# =========================================================
# 1. IMPORTS
# =========================================================

import os
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# 2. PATHS
# =========================================================

TRAFFIC_PATH = (
    "/Volumes/aeroflux/gold_layer/"
    "gold_h3_traffic"
)

MODEL_PATH = (
    "/Volumes/aeroflux/gold_layer/"
    "gold_h3_congestion/"
    "xgb_models"
)


# Managed Delta tables.
#
# Using tables here avoids creating another Delta root
# underneath an existing Gold Delta dataset.

FORECAST_TABLE = (
    "aeroflux.gold_layer."
    "xgb_congestion_forecast"
)

LATEST_TABLE = (
    "aeroflux.gold_layer."
    "xgb_congestion_latest"
)

METRICS_TABLE = (
    "aeroflux.gold_layer."
    "xgb_congestion_metrics"
)

IMPORTANCE_TABLE = (
    "aeroflux.gold_layer."
    "xgb_congestion_feature_importance"
)


# =========================================================
# 3. CONFIGURATION
# =========================================================

HORIZONS = [5, 15, 30]

TIME_STEP_SECONDS = 300

TRAIN_RATIO = 0.80

RANDOM_STATE = 42

# Ignore cells with almost no history.
MIN_ACTIVE_WINDOWS_PER_CELL = 4

# Safety limits for Databricks Free Edition.
MAX_GRID_ROWS = 5_000_000
MAX_MODEL_ROWS = 500_000


# =========================================================
# 4. READ GOLD H3 TRAFFIC
# =========================================================

traffic = (
    spark.read
    .format("delta")
    .load(TRAFFIC_PATH)
)


# =========================================================
# 5. SCHEMA VALIDATION
# =========================================================

required_columns = [
    "h3_res_8",
    "window_start",
    "window_end",
    "unique_aircraft_count",
    "observation_count",
    "inflow_unique_aircraft",
    "outflow_unique_aircraft",
    "net_aircraft_flow",
    "mean_altitude_m",
    "mean_velocity_mps"
]


missing_columns = [
    c
    for c in required_columns
    if c not in traffic.columns
]


if missing_columns:

    raise ValueError(
        "Missing required Gold traffic columns: "
        + ", ".join(missing_columns)
    )


# =========================================================
# 6. DEFENSIVE NORMALIZATION
# =========================================================

traffic = (
    traffic

    .select(
        *required_columns
    )

    .filter(
        F.col("h3_res_8").isNotNull()
        &
        F.col("window_start").isNotNull()
    )

    # Defensive duplicate protection.
    .groupBy(
        "h3_res_8",
        "window_start",
        "window_end"
    )

    .agg(

        F.max(
            "unique_aircraft_count"
        ).alias(
            "unique_aircraft_count"
        ),

        F.max(
            "observation_count"
        ).alias(
            "observation_count"
        ),

        F.max(
            "inflow_unique_aircraft"
        ).alias(
            "inflow_unique_aircraft"
        ),

        F.max(
            "outflow_unique_aircraft"
        ).alias(
            "outflow_unique_aircraft"
        ),

        F.max(
            "net_aircraft_flow"
        ).alias(
            "net_aircraft_flow"
        ),

        F.max(
            "mean_altitude_m"
        ).alias(
            "mean_altitude_m"
        ),

        F.max(
            "mean_velocity_mps"
        ).alias(
            "mean_velocity_mps"
        )
    )

    .withColumn(
        "window_epoch",
        F.col(
            "window_start"
        ).cast("long")
    )
)


# =========================================================
# 7. GLOBAL OBSERVED TIME AXIS
# =========================================================
#
# Only timestamps that actually exist somewhere in Gold
# are used.
#
# We deliberately do NOT manufacture global zero-data
# periods.
# =========================================================

time_dim = (
    traffic

    .select(
        "window_start",
        "window_epoch"
    )

    .distinct()
)


time_count = time_dim.count()


print(
    f"Observed global time windows: {time_count:,}"
)


if time_count < 10:

    raise ValueError(
        "Too few observed time windows for forecasting."
    )


# =========================================================
# 8. ELIGIBLE H3 CELLS
# =========================================================

cell_stats = (
    traffic

    .groupBy(
        "h3_res_8"
    )

    .agg(
        F.countDistinct(
            "window_start"
        ).alias(
            "active_window_count"
        )
    )

    .filter(
        F.col(
            "active_window_count"
        )
        >=
        MIN_ACTIVE_WINDOWS_PER_CELL
    )
)


cell_count = cell_stats.count()


print(
    f"Eligible H3 cells: {cell_count:,}"
)


if cell_count == 0:

    raise ValueError(
        "No H3 cells contain enough historical activity."
    )


# =========================================================
# 9. CONTROL GRID SIZE
# =========================================================

estimated_grid_rows = (
    cell_count
    *
    time_count
)


if estimated_grid_rows > MAX_GRID_ROWS:

    max_cells = max(
        100,
        MAX_GRID_ROWS // time_count
    )

    print(
        "Estimated grid is too large:"
        f" {estimated_grid_rows:,}"
    )

    print(
        f"Keeping top {max_cells:,} H3 cells "
        "by historical activity."
    )

    ranking_window = (
        Window
        .orderBy(
            F.col(
                "active_window_count"
            ).desc(),

            F.col(
                "h3_res_8"
            ).asc()
        )
    )

    cell_stats = (
        cell_stats

        .withColumn(
            "cell_rank",
            F.row_number().over(
                ranking_window
            )
        )

        .filter(
            F.col(
                "cell_rank"
            )
            <=
            max_cells
        )

        .drop(
            "cell_rank"
        )
    )

    cell_count = cell_stats.count()

    estimated_grid_rows = (
        cell_count
        *
        time_count
    )


print(
    f"Final grid estimate: "
    f"{estimated_grid_rows:,} rows"
)


# =========================================================
# 10. BUILD H3 × GLOBAL TIME GRID
# =========================================================
#
# Example:
#
# Cell A exists at 10:00 and 10:05
# Cell B exists at 10:05
#
# Gold modeling grid:
#
# Cell A 10:00 -> observed
# Cell A 10:05 -> observed
# Cell B 10:00 -> zero
# Cell B 10:05 -> observed
#
# This is appropriate when each global OpenSky snapshot
# represents the observed airspace state.
# =========================================================

regular = (
    cell_stats

    .select(
        "h3_res_8"
    )

    .crossJoin(
        time_dim
    )

    .join(
        traffic,
        [
            "h3_res_8",
            "window_start",
            "window_epoch"
        ],
        "left"
    )
)


# =========================================================
# 11. ZERO-TRAFFIC REPRESENTATION
# =========================================================

regular = (
    regular

    .withColumn(
        "observed_in_source",
        F.when(
            F.col(
                "observation_count"
            ).isNull(),
            0
        )
        .otherwise(
            1
        )
    )

    .withColumn(
        "unique_aircraft_count",
        F.coalesce(
            F.col(
                "unique_aircraft_count"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "observation_count",
        F.coalesce(
            F.col(
                "observation_count"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "inflow_unique_aircraft",
        F.coalesce(
            F.col(
                "inflow_unique_aircraft"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "outflow_unique_aircraft",
        F.coalesce(
            F.col(
                "outflow_unique_aircraft"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "net_aircraft_flow",
        F.coalesce(
            F.col(
                "net_aircraft_flow"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "mean_altitude_m",
        F.coalesce(
            F.col(
                "mean_altitude_m"
            ),
            F.lit(0.0)
        )
    )

    .withColumn(
        "mean_velocity_mps",
        F.coalesce(
            F.col(
                "mean_velocity_mps"
            ),
            F.lit(0.0)
        )
    )
)


# =========================================================
# 12. H3 TIME-SERIES WINDOW
# =========================================================

h3_window = (
    Window
    .partitionBy(
        "h3_res_8"
    )
    .orderBy(
        "window_epoch"
    )
)


# =========================================================
# 13. TIME FEATURES
# =========================================================

regular = (
    regular

    .withColumn(
        "hour_of_day",
        F.hour(
            "window_start"
        )
    )

    .withColumn(
        "day_of_week",
        F.dayofweek(
            "window_start"
        )
    )

    .withColumn(
        "minute_of_hour",
        F.minute(
            "window_start"
        )
    )

    .withColumn(
        "hour_sin",
        F.sin(
            2
            * F.lit(np.pi)
            * (
                F.col("hour_of_day")
                +
                F.col("minute_of_hour") / 60.0
            )
            / 24.0
        )
    )

    .withColumn(
        "hour_cos",
        F.cos(
            2
            * F.lit(np.pi)
            * (
                F.col("hour_of_day")
                +
                F.col("minute_of_hour") / 60.0
            )
            / 24.0
        )
    )

    .withColumn(
        "dow_sin",
        F.sin(
            2
            * F.lit(np.pi)
            * (
                F.col("day_of_week") - 1
            )
            / 7.0
        )
    )

    .withColumn(
        "dow_cos",
        F.cos(
            2
            * F.lit(np.pi)
            * (
                F.col("day_of_week") - 1
            )
            / 7.0
        )
    )
)


# =========================================================
# 14. VALIDATED LAG FEATURES
# =========================================================
#
# The lag is accepted only when the real timestamp distance
# equals the intended interval.
# =========================================================

lag_definitions = {
    1: 300,
    2: 600,
    3: 900,
    6: 1800,
    12: 3600
}


for lag, expected_seconds in lag_definitions.items():

    lag_epoch = F.lag(
        "window_epoch",
        lag
    ).over(
        h3_window
    )

    lag_traffic = F.lag(
        "unique_aircraft_count",
        lag
    ).over(
        h3_window
    )

    lag_inflow = F.lag(
        "inflow_unique_aircraft",
        lag
    ).over(
        h3_window
    )

    lag_outflow = F.lag(
        "outflow_unique_aircraft",
        lag
    ).over(
        h3_window
    )

    lag_velocity = F.lag(
        "mean_velocity_mps",
        lag
    ).over(
        h3_window
    )

    valid_lag = (
        F.col(
            "window_epoch"
        )
        -
        lag_epoch
        ==
        expected_seconds
    )

    regular = (
        regular

        .withColumn(
            f"traffic_lag_{lag}",
            F.when(
                valid_lag,
                lag_traffic
            )
        )

        .withColumn(
            f"inflow_lag_{lag}",
            F.when(
                valid_lag,
                lag_inflow
            )
        )

        .withColumn(
            f"outflow_lag_{lag}",
            F.when(
                valid_lag,
                lag_outflow
            )
        )

        .withColumn(
            f"velocity_lag_{lag}",
            F.when(
                valid_lag,
                lag_velocity
            )
        )
    )


# =========================================================
# 15. TIME-RANGE ROLLING FEATURES
# =========================================================

rolling_15 = (
    Window
    .partitionBy(
        "h3_res_8"
    )
    .orderBy(
        "window_epoch"
    )
    .rangeBetween(
        -900,
        -300
    )
)


rolling_30 = (
    Window
    .partitionBy(
        "h3_res_8"
    )
    .orderBy(
        "window_epoch"
    )
    .rangeBetween(
        -1800,
        -300
    )
)


rolling_60 = (
    Window
    .partitionBy(
        "h3_res_8"
    )
    .orderBy(
        "window_epoch"
    )
    .rangeBetween(
        -3600,
        -300
    )
)


regular = (
    regular

    .withColumn(
        "traffic_mean_15m",
        F.avg(
            "unique_aircraft_count"
        ).over(
            rolling_15
        )
    )

    .withColumn(
        "traffic_std_15m",
        F.stddev(
            "unique_aircraft_count"
        ).over(
            rolling_15
        )
    )

    .withColumn(
        "traffic_mean_30m",
        F.avg(
            "unique_aircraft_count"
        ).over(
            rolling_30
        )
    )

    .withColumn(
        "traffic_std_30m",
        F.stddev(
            "unique_aircraft_count"
        ).over(
            rolling_30
        )
    )

    .withColumn(
        "traffic_max_30m",
        F.max(
            "unique_aircraft_count"
        ).over(
            rolling_30
        )
    )

    .withColumn(
        "traffic_mean_60m",
        F.avg(
            "unique_aircraft_count"
        ).over(
            rolling_60
        )
    )

    .withColumn(
        "traffic_std_60m",
        F.stddev(
            "unique_aircraft_count"
        ).over(
            rolling_60
        )
    )
)


# =========================================================
# 16. TREND FEATURES
# =========================================================

regular = (
    regular

    .withColumn(
        "traffic_change_5m",
        F.col(
            "unique_aircraft_count"
        )
        -
        F.col(
            "traffic_lag_1"
        )
    )

    .withColumn(
        "traffic_change_15m",
        F.col(
            "unique_aircraft_count"
        )
        -
        F.col(
            "traffic_lag_3"
        )
    )

    .withColumn(
        "traffic_change_30m",
        F.col(
            "unique_aircraft_count"
        )
        -
        F.col(
            "traffic_lag_6"
        )
    )

    .withColumn(
        "flow_pressure",
        F.col(
            "inflow_unique_aircraft"
        )
        -
        F.col(
            "outflow_unique_aircraft"
        )
    )
)


# =========================================================
# 17. VALIDATED FUTURE TARGETS
# =========================================================
#
# IMPORTANT:
# t+15 is valid only when the actual future timestamp is
# exactly 15 minutes later.
# =========================================================

for horizon in HORIZONS:

    steps = horizon // 5

    future_epoch = F.lead(
        "window_epoch",
        steps
    ).over(
        h3_window
    )

    future_count = F.lead(
        "unique_aircraft_count",
        steps
    ).over(
        h3_window
    )

    regular = regular.withColumn(
        f"target_{horizon}m",
        F.when(
            (
                future_epoch
                -
                F.col("window_epoch")
            )
            ==
            horizon * 60,
            future_count
        )
    )


# =========================================================
# 18. HISTORY QUALITY FEATURE
# =========================================================

history_columns = [
    "traffic_lag_1",
    "traffic_lag_2",
    "traffic_lag_3",
    "traffic_lag_6",
    "traffic_lag_12",
    "traffic_mean_15m",
    "traffic_mean_30m",
    "traffic_mean_60m"
]


regular = regular.withColumn(
    "history_missing_count",
    sum(
        F.col(c)
        .isNull()
        .cast("int")
        for c in history_columns
    )
)


# =========================================================
# 19. FINAL FEATURE SET
# =========================================================
#
# IMPORTANT:
# observed_in_source is deliberately excluded.
#
# global_gap_minutes is deliberately excluded.
#
# Both were data-collection indicators rather than genuine
# traffic dynamics. Removing them prevents the model from
# learning source/grid availability instead of congestion.
# =========================================================

feature_columns = [

    # Current traffic
    "unique_aircraft_count",
    "observation_count",

    # Flow
    "inflow_unique_aircraft",
    "outflow_unique_aircraft",
    "net_aircraft_flow",
    "flow_pressure",

    # State
    "mean_altitude_m",
    "mean_velocity_mps",

    # Validated lags
    "traffic_lag_1",
    "traffic_lag_2",
    "traffic_lag_3",
    "traffic_lag_6",
    "traffic_lag_12",

    "inflow_lag_1",
    "inflow_lag_3",
    "inflow_lag_6",

    "outflow_lag_1",
    "outflow_lag_3",
    "outflow_lag_6",

    "velocity_lag_1",
    "velocity_lag_3",
    "velocity_lag_6",

    # Rolling traffic
    "traffic_mean_15m",
    "traffic_std_15m",

    "traffic_mean_30m",
    "traffic_std_30m",
    "traffic_max_30m",

    "traffic_mean_60m",
    "traffic_std_60m",

    # Trend
    "traffic_change_5m",
    "traffic_change_15m",
    "traffic_change_30m",

    # History quality
    "history_missing_count",

    # Cyclic time
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos"
]


# =========================================================
# 20. MODEL DATASET
# =========================================================

candidate_columns = [
    "h3_res_8",
    "window_start",
    *feature_columns,
    "target_5m",
    "target_15m",
    "target_30m"
]


model_sdf = regular.select(
    *candidate_columns
)


candidate_count = model_sdf.count()


print(
    f"Candidate modeling rows: {candidate_count:,}"
)


# =========================================================
# 21. PANDAS MEMORY CONTROL
# =========================================================
#
# Preserve signal rows:
#
#   current traffic > 0
#   OR future traffic > 0
#
# Then retain up to 2x as many zero-only examples.
# =========================================================

signal_condition = (
    (F.col("unique_aircraft_count") > 0)
    |
    (
        F.coalesce(
            F.col("target_5m"),
            F.lit(0.0)
        ) > 0
    )
    |
    (
        F.coalesce(
            F.col("target_15m"),
            F.lit(0.0)
        ) > 0
    )
    |
    (
        F.coalesce(
            F.col("target_30m"),
            F.lit(0.0)
        ) > 0
    )
)


signal_sdf = (
    model_sdf
    .filter(
        signal_condition
    )
)


zero_sdf = (
    model_sdf
    .filter(
        ~signal_condition
    )
)


signal_count = signal_sdf.count()
zero_count = zero_sdf.count()


print(
    f"Signal rows: {signal_count:,}"
)

print(
    f"Zero-only rows: {zero_count:,}"
)


max_zero_rows = min(
    zero_count,
    signal_count * 2
)


if max_zero_rows > 0:

    zero_sdf = (
        zero_sdf

        .withColumn(
            "_sample_key",
            F.xxhash64(
                "h3_res_8",
                "window_start"
            )
        )

        .orderBy(
            "_sample_key"
        )

        .limit(
            int(max_zero_rows)
        )

        .drop(
            "_sample_key"
        )
    )


if signal_count > 0:

    model_sdf = (
        signal_sdf
        .unionByName(
            zero_sdf
        )
    )

else:

    model_sdf = zero_sdf


model_count = model_sdf.count()


if model_count > MAX_MODEL_ROWS:

    model_sdf = (
        model_sdf

        .withColumn(
            "_sample_key",
            F.xxhash64(
                "h3_res_8",
                "window_start"
            )
        )

        .orderBy(
            "_sample_key"
        )

        .limit(
            MAX_MODEL_ROWS
        )

        .drop(
            "_sample_key"
        )
    )


# =========================================================
# 22. COLLECT TO PANDAS
# =========================================================

model_df = (
    model_sdf
    .toPandas()
)


model_df["window_start"] = pd.to_datetime(
    model_df["window_start"]
)


model_df = (
    model_df
    .sort_values(
        "window_start"
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Final pandas modeling rows: "
    f"{len(model_df):,}"
)


# =========================================================
# 23. COMPLETE LATEST STATE
# =========================================================
#
# This is collected separately so model sampling never
# removes a current H3 cell from the inference output.
# =========================================================

latest_time = (
    regular
    .select(
        F.max(
            "window_start"
        ).alias(
            "latest_time"
        )
    )
    .collect()[0]["latest_time"]
)


latest_sdf = (
    regular

    .filter(
        F.col(
            "window_start"
        )
        ==
        latest_time
    )

    .filter(
        (F.col(
            "unique_aircraft_count"
        ) > 0)
        |
        (
            F.coalesce(
                F.col(
                    "traffic_mean_60m"
                ),
                F.lit(0.0)
            )
            > 0
        )
    )

    .select(
        "h3_res_8",
        "window_start",
        *feature_columns
    )
)


latest_df = latest_sdf.toPandas()


latest_df["window_start"] = pd.to_datetime(
    latest_df["window_start"]
)


print(
    f"Latest observed window: {latest_time}"
)

print(
    f"Latest forecastable H3 cells: "
    f"{len(latest_df):,}"
)


# =========================================================
# 24. CHRONOLOGICAL SPLIT
# =========================================================

unique_times = (
    model_df[
        "window_start"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(
        drop=True
    )
)


if len(unique_times) < 10:

    raise ValueError(
        "Too few modeling time windows."
    )


cutoff_index = int(
    len(unique_times)
    *
    TRAIN_RATIO
)


cutoff_time = unique_times[
    cutoff_index
]


train_base = (
    model_df[
        model_df["window_start"]
        <
        cutoff_time
    ]
    .copy()
)


test_base = (
    model_df[
        model_df["window_start"]
        >=
        cutoff_time
    ]
    .copy()
)


print(
    f"Train rows: {len(train_base):,}"
)

print(
    f"Test rows : {len(test_base):,}"
)

print(
    f"Training cutoff: {cutoff_time}"
)


if len(train_base) < 100:

    raise ValueError(
        "Fewer than 100 training rows."
    )

if len(test_base) < 20:

    raise ValueError(
        "Fewer than 20 test rows."
    )


# =========================================================
# 25. MODEL CONFIGURATION
# =========================================================
#
# count:poisson models non-negative aircraft counts and
# avoids negative traffic predictions.
# =========================================================

xgb_params = dict(

    objective="count:poisson",

    n_estimators=350,

    max_depth=5,

    learning_rate=0.05,

    subsample=0.85,

    colsample_bytree=0.85,

    min_child_weight=3,

    reg_alpha=0.05,

    reg_lambda=1.0,

    max_delta_step=0.7,

    tree_method="hist",

    random_state=RANDOM_STATE,

    n_jobs=-1
)


# =========================================================
# 26. HELPERS
# =========================================================

def make_model():

    return xgb.XGBRegressor(
        **xgb_params
    )


def fit_imputer(data):

    values = {}

    for column in feature_columns:

        series = pd.to_numeric(
            data[column],
            errors="coerce"
        )

        median_value = series.median()

        if pd.isna(
            median_value
        ):

            median_value = 0.0

        values[column] = float(
            median_value
        )

    return values


def apply_imputer(
    data,
    imputer
):

    result = data[
        feature_columns
    ].copy()

    for column in feature_columns:

        result[column] = pd.to_numeric(
            result[column],
            errors="coerce"
        )

        result[column] = (
            result[column]

            .replace(
                [np.inf, -np.inf],
                np.nan
            )

            .fillna(
                imputer[column]
            )
        )

    return result


# =========================================================
# 27. TRAIN MODELS
# =========================================================

all_prediction_outputs = []

all_metric_outputs = []

all_importance_outputs = []

latest_outputs = []

trained_models = {}


for horizon in HORIZONS:

    print()
    print("=" * 70)

    print(
        f"TRAINING XGBOOST — {horizon} MINUTE HORIZON"
    )

    print("=" * 70)


    target = (
        f"target_{horizon}m"
    )


    train_df = (
        train_base
        .dropna(
            subset=[
                target
            ]
        )
        .copy()
    )


    test_df = (
        test_base
        .dropna(
            subset=[
                target
            ]
        )
        .copy()
    )


    print(
        f"Usable train rows: "
        f"{len(train_df):,}"
    )

    print(
        f"Usable test rows : "
        f"{len(test_df):,}"
    )


    if len(train_df) < 100:

        print(
            f"Skipping {horizon}m — "
            "insufficient training examples."
        )

        continue


    if len(test_df) < 20:

        print(
            f"Skipping {horizon}m — "
            "insufficient test examples."
        )

        continue


    # -----------------------------------------------------
    # TRAIN / TEST FEATURES
    # -----------------------------------------------------

    train_imputer = fit_imputer(
        train_df
    )


    X_train = apply_imputer(
        train_df,
        train_imputer
    )


    X_test = apply_imputer(
        test_df,
        train_imputer
    )


    y_train = (
        train_df[target]
        .astype(float)
        .values
    )


    y_test = (
        test_df[target]
        .astype(float)
        .values
    )


    # -----------------------------------------------------
    # PERSISTENCE BASELINE
    # -----------------------------------------------------

    baseline_prediction = (
        test_df[
            "unique_aircraft_count"
        ]
        .astype(float)
        .values
    )


    baseline_mae = (
        mean_absolute_error(
            y_test,
            baseline_prediction
        )
    )


    baseline_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            baseline_prediction
        )
    )


    # -----------------------------------------------------
    # EVALUATION MODEL
    # -----------------------------------------------------

    evaluation_model = make_model()


    evaluation_model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_test,
                y_test
            )
        ],
        verbose=False
    )


    test_prediction = (
        evaluation_model
        .predict(
            X_test
        )
    )


    test_prediction = np.maximum(
        test_prediction,
        0.0
    )


    mae = (
        mean_absolute_error(
            y_test,
            test_prediction
        )
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )


    improvement_pct = (
        (
            baseline_mae
            -
            mae
        )
        /
        baseline_mae
        *
        100.0
        if baseline_mae > 0
        else 0.0
    )


    print(
        f"XGBoost MAE      : {mae:.4f}"
    )

    print(
        f"XGBoost RMSE     : {rmse:.4f}"
    )

    print(
        f"Persistence MAE  : {baseline_mae:.4f}"
    )

    print(
        f"Persistence RMSE : {baseline_rmse:.4f}"
    )

    print(
        f"Improvement       : "
        f"{improvement_pct:.2f}%"
    )


    # -----------------------------------------------------
    # EVALUATION OUTPUT
    # -----------------------------------------------------

    prediction_output = pd.DataFrame({

        "h3_res_8":
            test_df[
                "h3_res_8"
            ].values,

        "prediction_time":
            test_df[
                "window_start"
            ].values,

        "horizon_minutes":
            horizon,

        "actual_aircraft_count":
            y_test,

        "predicted_aircraft_count":
            test_prediction,

        "predicted_aircraft_count_rounded":
            np.rint(
                test_prediction
            ).astype(int),

        "persistence_prediction":
            baseline_prediction,

        "absolute_error":
            np.abs(
                y_test
                -
                test_prediction
            )
    })


    all_prediction_outputs.append(
        prediction_output
    )


    # -----------------------------------------------------
    # METRICS
    # -----------------------------------------------------

    all_metric_outputs.append({

        "model":
            "XGBoost",

        "horizon_minutes":
            horizon,

        "train_rows":
            len(train_df),

        "test_rows":
            len(test_df),

        "mae":
            float(mae),

        "rmse":
            float(rmse),

        "baseline_mae":
            float(
                baseline_mae
            ),

        "baseline_rmse":
            float(
                baseline_rmse
            ),

        "improvement_over_persistence_pct":
            float(
                improvement_pct
            ),

        "train_start":
            train_df[
                "window_start"
            ].min(),

        "train_end":
            train_df[
                "window_start"
            ].max(),

        "test_start":
            test_df[
                "window_start"
            ].min(),

        "test_end":
            test_df[
                "window_start"
            ].max(),

        "feature_count":
            len(
                feature_columns
            )
    })


    # -----------------------------------------------------
    # FEATURE IMPORTANCE
    # -----------------------------------------------------

    for feature, importance in zip(
        feature_columns,
        evaluation_model.feature_importances_
    ):

        all_importance_outputs.append({

            "horizon_minutes":
                horizon,

            "feature":
                feature,

            "importance":
                float(
                    importance
                )
        })


    # -----------------------------------------------------
    # FINAL MODEL
    #
    # Refit on all available usable rows after evaluation.
    # -----------------------------------------------------

    full_df = (
        model_df
        .dropna(
            subset=[
                target
            ]
        )
        .copy()
    )


    final_imputer = fit_imputer(
        full_df
    )


    X_full = apply_imputer(
        full_df,
        final_imputer
    )


    y_full = (
        full_df[target]
        .astype(float)
        .values
    )


    final_model = make_model()


    final_model.fit(
        X_full,
        y_full,
        verbose=False
    )


    # -----------------------------------------------------
    # SAVE MODEL
    # -----------------------------------------------------

    os.makedirs(
        MODEL_PATH,
        exist_ok=True
    )


    model_file = (
        MODEL_PATH
        +
        f"/xgboost_{horizon}m_final.json"
    )


    final_model.save_model(
        model_file
    )


    trained_models[
        horizon
    ] = (
        final_model,
        final_imputer
    )


    # -----------------------------------------------------
    # LATEST FORECAST
    # -----------------------------------------------------

    if not latest_df.empty:

        X_latest = apply_imputer(
            latest_df,
            final_imputer
        )


        latest_prediction = (
            final_model
            .predict(
                X_latest
            )
        )


        latest_prediction = np.maximum(
            latest_prediction,
            0.0
        )


        latest_output = pd.DataFrame({

            "h3_res_8":
                latest_df[
                    "h3_res_8"
                ].values,

            "prediction_time":
                latest_df[
                    "window_start"
                ].values,

            "forecast_time":
                latest_df[
                    "window_start"
                ].values
                +
                pd.to_timedelta(
                    horizon,
                    unit="m"
                ),

            "horizon_minutes":
                horizon,

            "current_aircraft_count":
                latest_df[
                    "unique_aircraft_count"
                ].values,

            "predicted_aircraft_count":
                latest_prediction,

            "predicted_aircraft_count_rounded":
                np.rint(
                    latest_prediction
                ).astype(int),

            "forecast_delta":
                (
                    latest_prediction
                    -
                    latest_df[
                        "unique_aircraft_count"
                    ].values
                )
        })


        latest_outputs.append(
            latest_output
        )


# =========================================================
# 28. WRITE EVALUATION FORECASTS
# =========================================================

if all_prediction_outputs:

    forecast_pdf = pd.concat(
        all_prediction_outputs,
        ignore_index=True
    )


    forecast_spark = (
        spark.createDataFrame(
            forecast_pdf
        )
    )


    (
        forecast_spark
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            FORECAST_TABLE
        )
    )


    print(
        f"Saved forecast table: "
        f"{FORECAST_TABLE}"
    )


# =========================================================
# 29. WRITE LATEST FORECAST
# =========================================================

if latest_outputs:

    latest_pdf = pd.concat(
        latest_outputs,
        ignore_index=True
    )


    latest_spark = (
        spark.createDataFrame(
            latest_pdf
        )
    )


    (
        latest_spark
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            LATEST_TABLE
        )
    )


    print(
        f"Saved latest forecast: "
        f"{LATEST_TABLE}"
    )


# =========================================================
# 30. WRITE METRICS
# =========================================================

if all_metric_outputs:

    metrics_pdf = (
        pd.DataFrame(
            all_metric_outputs
        )
    )


    metrics_spark = (
        spark.createDataFrame(
            metrics_pdf
        )
    )


    (
        metrics_spark
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            METRICS_TABLE
        )
    )


    print(
        f"Saved metrics table: "
        f"{METRICS_TABLE}"
    )


    display(
        metrics_spark
        .orderBy(
            "horizon_minutes"
        )
    )


# =========================================================
# 31. WRITE FEATURE IMPORTANCE
# =========================================================

if all_importance_outputs:

    importance_pdf = (
        pd.DataFrame(
            all_importance_outputs
        )
    )


    importance_spark = (
        spark.createDataFrame(
            importance_pdf
        )
    )


    (
        importance_spark
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            IMPORTANCE_TABLE
        )
    )


    print(
        f"Saved feature importance: "
        f"{IMPORTANCE_TABLE}"
    )


    display(
        importance_spark
        .orderBy(
            "horizon_minutes",
            F.col(
                "importance"
            ).desc()
        )
    )


# =========================================================
# 32. FINAL STATUS
# =========================================================

print()
print("=" * 70)
print("AEROFLUX XGBOOST FORECASTING COMPLETE")
print("=" * 70)

print(
    "Models trained:",
    sorted(
        trained_models.keys()
    )
)

print(
    "Forecast table:",
    FORECAST_TABLE
)

print(
    "Latest forecast:",
    LATEST_TABLE
)

print(
    "Metrics table:",
    METRICS_TABLE
)

print(
    "Feature importance:",
    IMPORTANCE_TABLE
)

print(
    "Model files:",
    MODEL_PATH
)

print("=" * 70)

Observed global time windows: 103
Eligible H3 cells: 425
Final grid estimate: 43,775 rows
Candidate modeling rows: 43,775
Signal rows: 7,016
Zero-only rows: 36,759
Final pandas modeling rows: 21,048
Latest observed window: 2026-09-27 12:05:00
Latest forecastable H3 cells: 72
Train rows: 16,235
Test rows : 4,813
Training cutoff: 2026-09-27 07:45:00

TRAINING XGBOOST — 5 MINUTE HORIZON
Usable train rows: 13,962
Usable test rows : 4,273
XGBoost MAE      : 0.2815
XGBoost RMSE     : 0.4451
Persistence MAE  : 0.3499
Persistence RMSE : 0.6159
Improvement       : 19.53%

TRAINING XGBOOST — 15 MINUTE HORIZON
Usable train rows: 9,861
Usable test rows : 3,028
XGBoost MAE      : 0.3357
XGBoost RMSE     : 0.4823
Persistence MAE  : 0.3438
Persistence RMSE : 0.6090
Improvement       : 2.35%

TRAINING XGBOOST — 30 MINUTE HORIZON
Usable train rows: 5,107
Usable test rows : 1,585


/databricks/python/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


XGBoost MAE      : 0.3269
XGBoost RMSE     : 0.4965
Persistence MAE  : 0.3457
Persistence RMSE : 0.6194
Improvement       : 5.46%


/databricks/python/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Saved forecast table: aeroflux.gold_layer.xgb_congestion_forecast
Saved latest forecast: aeroflux.gold_layer.xgb_congestion_latest
Saved metrics table: aeroflux.gold_layer.xgb_congestion_metrics


model,horizon_minutes,train_rows,test_rows,mae,rmse,baseline_mae,baseline_rmse,improvement_over_persistence_pct,train_start,train_end,test_start,test_end,feature_count
XGBoost,5,13962,4273,0.2815454672518991,0.4450789382856805,0.3498712848116078,0.6159210698897963,19.528844042316734,2026-09-19T08:10:00.000Z,2026-09-27T07:40:00.000Z,2026-09-27T07:45:00.000Z,2026-09-27T12:00:00.000Z,37
XGBoost,15,9861,3028,0.3357153580517261,0.4823012850341906,0.3437912813738441,0.6089924979961476,2.3490774082010972,2026-09-19T12:00:00.000Z,2026-09-27T07:40:00.000Z,2026-09-27T07:45:00.000Z,2026-09-27T10:00:00.000Z,37
XGBoost,30,5107,1585,0.3268801701569501,0.49653072274855087,0.34574132492113563,0.6193514466851926,5.455279252050015,2026-09-23T17:45:00.000Z,2026-09-27T07:40:00.000Z,2026-09-27T07:45:00.000Z,2026-09-27T09:45:00.000Z,37


Saved feature importance: aeroflux.gold_layer.xgb_congestion_feature_importance


horizon_minutes,feature,importance
5,observation_count,0.1609870046377182
5,unique_aircraft_count,0.07676298916339874
5,traffic_lag_12,0.06193103641271591
5,traffic_std_60m,0.047857627272605896
5,dow_cos,0.04321670904755592
5,hour_sin,0.037536974996328354
5,traffic_mean_60m,0.03466682881116867
5,mean_velocity_mps,0.03417880833148956
5,hour_cos,0.032498180866241455
5,traffic_lag_3,0.030340375378727913



AEROFLUX XGBOOST FORECASTING COMPLETE
Models trained: [5, 15, 30]
Forecast table: aeroflux.gold_layer.xgb_congestion_forecast
Latest forecast: aeroflux.gold_layer.xgb_congestion_latest
Metrics table: aeroflux.gold_layer.xgb_congestion_metrics
Feature importance: aeroflux.gold_layer.xgb_congestion_feature_importance
Model files: /Volumes/aeroflux/gold_layer/gold_h3_congestion/xgb_models


# **Fuel/CO2**

In [0]:
# =========================================================
# AEROFLUX - FUEL / CO2 ESTIMATION
# =========================================================
#
# SOURCE:
#   gold_holding_events
#   gold_4d_trajectory
#
# EXTERNAL MODELS:
#   OpenSky aircraft metadata
#   OpenAP aircraft performance model
#
# OUTPUT:
#   /Volumes/aeroflux/gold_layer/
#       gold_holding_events/
#           fuel_co2_report/
#
# =========================================================

# =========================================================
# 1. IMPORTS
# =========================================================

import os
import zipfile
import requests

import pandas as pd

from pyspark.sql import functions as F


# =========================================================
# 2. PATHS
# =========================================================

HOLDING_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_holding_events"
)

TRAJECTORY_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_4d_trajectory"
)

OUTPUT_PATH = (
    "/Volumes/aeroflux/gold_layer/fuel_co2_report"
)


# =========================================================
# 3. ICAO CO2 CONVERSION
# =========================================================
#
# ICAO CORSIA:
#
# Jet-A / Jet-A1:
# 3.16 kg CO2 / kg fuel
#
# =========================================================

CO2_KG_PER_KG_FUEL = 3.16


# =========================================================
# 4. READ HOLDING EVENTS
# =========================================================

holding = (
    spark.read
    .format("delta")
    .load(HOLDING_PATH)
)


holding_count = holding.count()


print(
    f"Holding events available: {holding_count}"
)


# =========================================================
# 5. HANDLE ZERO-HOLDING CASE
# =========================================================
#
# Your current small dataset produced 0 holding events.
# In that case we still create a stable report schema.
# =========================================================

if holding_count == 0:

    empty_schema = (
        "holding_event_id string, "
        "icao24 string, "
        "aircraft_type string, "
        "holding_duration_minutes double, "
        "estimated_hold_fuel_kg double, "
        "estimated_baseline_fuel_kg double, "
        "estimated_incremental_fuel_kg double, "
        "estimated_incremental_co2_kg double, "
        "fuel_model string, "
        "fuel_model_status string, "
        "co2_conversion_factor_kg_per_kg_fuel double"
    )

    empty_df = spark.createDataFrame(
        [],
        empty_schema
    )

    (
        empty_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .save(
            OUTPUT_PATH
        )
    )

    print(
        "No holding events detected. "
        "Fuel/CO2 report created with empty schema."
    )

else:

    # =====================================================
    # 6. DOWNLOAD OPENSKY AIRCRAFT METADATA
    # =====================================================
    #
    # OpenSky currently provides this database as a CSV
    # dataset. The database is not guaranteed to be current.
    #
    # =====================================================

    ZIP_PATH = (
        "/tmp/aircraftDatabase.zip"
    )

    CSV_PATH = (
        "/tmp/aircraftDatabase.csv"
    )

    METADATA_URL = (
        "https://opensky-network.org/"
        "datasets/metadata/"
        "aircraftDatabase.zip"
    )


    if not os.path.exists(CSV_PATH):

        response = requests.get(
            METADATA_URL,
            timeout=120
        )

        response.raise_for_status()

        with open(
            ZIP_PATH,
            "wb"
        ) as f:

            f.write(
                response.content
            )


        with zipfile.ZipFile(
            ZIP_PATH,
            "r"
        ) as z:

            csv_members = [
                name
                for name in z.namelist()
                if name.lower().endswith(".csv")
            ]

            if not csv_members:

                raise RuntimeError(
                    "No CSV found inside "
                    "OpenSky aircraft metadata archive."
                )

            z.extract(
                csv_members[0],
                "/tmp"
            )

            extracted_csv = os.path.join(
                "/tmp",
                csv_members[0]
            )

            if extracted_csv != CSV_PATH:

                os.rename(
                    extracted_csv,
                    CSV_PATH
                )


    # =====================================================
    # 7. READ METADATA
    # =====================================================

    metadata = (
        spark.read
        .option(
            "header",
            True
        )
        .option(
            "inferSchema",
            False
        )
        .csv(
            CSV_PATH
        )
    )


    # =====================================================
    # 8. IDENTIFY AIRCRAFT TYPE COLUMN
    # =====================================================

    metadata_columns = [
        c.lower()
        for c in metadata.columns
    ]


    type_candidates = [
        "typecode",
        "icaoaircrafttype",
        "model"
    ]


    type_column = None


    for candidate in type_candidates:

        if candidate.lower() in metadata_columns:

            type_column = metadata.columns[
                metadata_columns.index(
                    candidate.lower()
                )
            ]

            break


    if type_column is None:

        raise ValueError(
            "Could not find aircraft type column "
            "in OpenSky metadata."
        )


    # =====================================================
    # 9. IDENTIFY ICAO24 COLUMN
    # =====================================================

    icao_column = None


    for candidate in [
        "icao24",
        "icao"
    ]:

        if candidate.lower() in metadata_columns:

            icao_column = metadata.columns[
                metadata_columns.index(
                    candidate.lower()
                )
            ]

            break


    if icao_column is None:

        raise ValueError(
            "Could not find ICAO24 column "
            "in OpenSky metadata."
        )


    aircraft_metadata = (
        metadata
        .select(
            F.lower(
                F.trim(
                    F.col(
                        icao_column
                    )
                )
            ).alias(
                "icao24"
            ),

            F.upper(
                F.trim(
                    F.col(
                        type_column
                    )
                )
            ).alias(
                "aircraft_type"
            )
        )
        .dropDuplicates(
            ["icao24"]
        )
    )


    # =====================================================
    # 10. READ 4D TRAJECTORY
    # =====================================================

    trajectory = (
        spark.read
        .format("delta")
        .load(
            TRAJECTORY_PATH
        )
    )


    # =====================================================
    # 11. JOIN HOLDING EVENTS TO TRAJECTORY
    # =====================================================
    #
    # Each observed state inside the holding-event interval
    # is used for fuel-flow estimation.
    #
    # =====================================================

    event_observations = (
        trajectory.alias("t")
        .join(
            holding.alias("h"),

            (
                (F.col("t.icao24") ==
                 F.col("h.icao24"))

                &

                (
                    F.col(
                        "t.trajectory_segment_id"
                    )
                    ==
                    F.col(
                        "h.trajectory_segment_id"
                    )
                )

                &

                (
                    F.col("t.event_time")
                    >=
                    F.col("h.holding_start")
                )

                &

                (
                    F.col("t.event_time")
                    <=
                    F.col("h.holding_end")
                )
            ),

            "inner"
        )

        .select(
            F.col(
                "h.holding_event_id"
            ),

            F.col(
                "h.icao24"
            ),

            F.col(
                "h.airport"
            ),

            F.col(
                "h.holding_start"
            ),

            F.col(
                "h.holding_end"
            ),

            F.col(
                "h.duration_minutes"
            ),

            F.col(
                "t.event_time"
            ),

            F.col(
                "t.time_delta_seconds"
            ),

            F.col(
                "t.trajectory_step_valid"
            ),

            F.col(
                "t.velocity_mps"
            ),

            F.col(
                "t.baro_altitude_m"
            ),

            F.col(
                "t.vertical_rate_mps"
            ),

            F.col(
                "t.acceleration_mps2"
            )
        )
    )


    # =====================================================
    # 12. JOIN AIRCRAFT TYPE
    # =====================================================

    event_observations = (
        event_observations
        .join(
            aircraft_metadata,
            "icao24",
            "left"
        )
    )


    # =====================================================
    # 13. COLLECT SMALL EVENT DATASET
    # =====================================================

    event_pdf = (
        event_observations
        .orderBy(
            "holding_event_id",
            "event_time"
        )
        .toPandas()
    )


    # =====================================================
    # 14. OPENAP FUEL CALCULATION
    # =====================================================

    from openap import (
        FuelFlow,
        prop
    )


    MPS_TO_KNOTS = (
        1.9438444924406
    )

    M_TO_FEET = (
        3.280839895
    )

    MPS_TO_FPM = (
        196.8503937008
    )


    # Cache loaded aircraft properties/models.
    fuel_models = {}
    aircraft_properties = {}


    def calculate_event_fuel(
        event_df
    ):

        aircraft_type = (
            event_df[
                "aircraft_type"
            ]
            .dropna()
            .astype(str)
            .str.upper()
            .iloc[0]
            if event_df[
                "aircraft_type"
            ].notna().any()
            else None
        )


        # -------------------------------------------------
        # Unsupported/missing aircraft type
        # -------------------------------------------------

        if not aircraft_type:

            return {
                "estimated_hold_fuel_kg": None,
                "estimated_baseline_fuel_kg": None,
                "estimated_incremental_fuel_kg": None,
                "fuel_model_status":
                    "MISSING_AIRCRAFT_TYPE"
            }


        try:

            if aircraft_type not in fuel_models:

                fuel_models[
                    aircraft_type
                ] = FuelFlow(
                    aircraft_type
                )

                aircraft_properties[
                    aircraft_type
                ] = prop.aircraft(
                    aircraft_type
                )


            fuel_model = fuel_models[
                aircraft_type
            ]

            aircraft = aircraft_properties[
                aircraft_type
            ]


            mtow = float(
                aircraft["mtow"]
            )


            # Conservative representative mass
            # assumption for an observed holding-like
            # segment.
            initial_mass_kg = (
                0.75
                *
                mtow
            )


        except Exception:

            return {
                "estimated_hold_fuel_kg": None,
                "estimated_baseline_fuel_kg": None,
                "estimated_incremental_fuel_kg": None,
                "fuel_model_status":
                    "OPENAP_UNSUPPORTED_TYPE"
            }


        # -------------------------------------------------
        # Time-order the trajectory
        # -------------------------------------------------

        event_df = (
            event_df
            .sort_values(
                "event_time"
            )
            .copy()
        )


        mass_kg = (
            initial_mass_kg
        )

        total_hold_fuel = 0.0

        valid_rows = []


        # -------------------------------------------------
        # Integrate fuel flow over valid state intervals
        # -------------------------------------------------

        for _, row in event_df.iterrows():

            if not bool(
                row[
                    "trajectory_step_valid"
                ]
            ):

                continue


            dt = row[
                "time_delta_seconds"
            ]


            velocity = row[
                "velocity_mps"
            ]

            altitude = row[
                "baro_altitude_m"
            ]

            vertical_rate = row[
                "vertical_rate_mps"
            ]


            if pd.isna(dt):
                continue

            if pd.isna(velocity):
                continue

            if pd.isna(altitude):
                continue


            if dt <= 0:
                continue


            # OpenAP expects:
            #
            # TAS      → knots
            # altitude → feet
            # VS       → feet/min
            #
            # OpenSky gives:
            #
            # velocity → m/s
            # altitude → m
            # vertical → m/s
            #
            tas_knots = max(
                50.0,
                float(velocity)
                *
                MPS_TO_KNOTS
            )


            altitude_ft = max(
                1000.0,
                float(altitude)
                *
                M_TO_FEET
            )


            vs_fpm = 0.0


            if not pd.isna(
                vertical_rate
            ):

                vs_fpm = (
                    float(vertical_rate)
                    *
                    MPS_TO_FPM
                )


            try:

                ff_kg_s = float(
                    fuel_model.enroute(
                        mass=mass_kg,
                        tas=tas_knots,
                        alt=altitude_ft,
                        vs=vs_fpm,
                        acc=0
                    )
                )


            except Exception:

                continue


            fuel_step = (
                ff_kg_s
                *
                float(dt)
            )


            fuel_step = max(
                0.0,
                fuel_step
            )


            total_hold_fuel += (
                fuel_step
            )


            mass_kg = max(
                mass_kg
                -
                fuel_step,
                0.50 * mtow
            )


            valid_rows.append({
                "fuel_flow_kg_s":
                    ff_kg_s,

                "fuel_step_kg":
                    fuel_step
            })


        # -------------------------------------------------
        # No usable intervals
        # -------------------------------------------------

        if not valid_rows:

            return {
                "estimated_hold_fuel_kg": None,
                "estimated_baseline_fuel_kg": None,
                "estimated_incremental_fuel_kg": None,
                "fuel_model_status":
                    "NO_VALID_TRAJECTORY_INTERVALS"
            }


        # -------------------------------------------------
        # Simplified counterfactual baseline
        #
        # Baseline:
        # direct displacement / typical observed speed
        #
        # This intentionally does NOT claim to model
        # airline dispatch fuel or an actual ATC counterfactual.
        # -------------------------------------------------

        # Approximate average fuel flow
        average_ff = (
            sum(
                x["fuel_flow_kg_s"]
                for x in valid_rows
            )
            /
            len(valid_rows)
        )


        valid_speed = (
            event_df[
                "velocity_mps"
            ]
            .dropna()
        )


        mean_speed = (
            float(
                valid_speed.mean()
            )
            if len(valid_speed) > 0
            else 1.0
        )


        mean_speed = max(
            mean_speed,
            1.0
        )


        # Event's endpoint displacement is retained
        # from the Gold holding event.
        #
        # The holding table does not contain it in this
        # dataframe, so derive it from start/end positions
        # outside this function only if supplied.
        #
        # Here use observed event duration as the conservative
        # baseline-time upper bound.
        observed_duration = (
            (
                event_df[
                    "event_time"
                ].max()
                -
                event_df[
                    "event_time"
                ].min()
            ).total_seconds()
        )


        baseline_duration = min(
            observed_duration,
            max(
                60.0,
                observed_duration
                *
                0.20
            )
        )


        baseline_fuel = (
            average_ff
            *
            baseline_duration
        )


        incremental_fuel = max(
            0.0,
            total_hold_fuel
            -
            baseline_fuel
        )


        return {

            "estimated_hold_fuel_kg":
                float(
                    total_hold_fuel
                ),

            "estimated_baseline_fuel_kg":
                float(
                    baseline_fuel
                ),

            "estimated_incremental_fuel_kg":
                float(
                    incremental_fuel
                ),

            "fuel_model_status":
                "OPENAP_ESTIMATE"
        }


    # =====================================================
    # 15. CALCULATE EVENT-LEVEL REPORT
    # =====================================================

    report_rows = []


    for event_id, event_group in (
        event_pdf
        .groupby(
            "holding_event_id"
        )
    ):

        fuel_result = (
            calculate_event_fuel(
                event_group
            )
        )


        aircraft_type = (
            event_group[
                "aircraft_type"
            ]
            .dropna()
            .astype(str)
            .iloc[0]
            if event_group[
                "aircraft_type"
            ].notna().any()
            else None
        )


        incremental_fuel = (
            fuel_result[
                "estimated_incremental_fuel_kg"
            ]
        )


        incremental_co2 = (
            incremental_fuel
            *
            CO2_KG_PER_KG_FUEL
            if incremental_fuel
            is not None
            else None
        )


        report_rows.append({

            "holding_event_id":
                event_id,

            "icao24":
                event_group[
                    "icao24"
                ].iloc[0],

            "aircraft_type":
                aircraft_type,

            "holding_duration_minutes":
                float(
                    event_group[
                        "duration_minutes"
                    ].iloc[0]
                ),

            "estimated_hold_fuel_kg":
                fuel_result[
                    "estimated_hold_fuel_kg"
                ],

            "estimated_baseline_fuel_kg":
                fuel_result[
                    "estimated_baseline_fuel_kg"
                ],

            "estimated_incremental_fuel_kg":
                incremental_fuel,

            "estimated_incremental_co2_kg":
                incremental_co2,

            "fuel_model":
                "OpenAP",

            "fuel_model_status":
                fuel_result[
                    "fuel_model_status"
                ],

            "co2_conversion_factor_kg_per_kg_fuel":
                CO2_KG_PER_KG_FUEL
        })


    report_pdf = pd.DataFrame(
        report_rows
    )


    # =====================================================
    # 16. WRITE EVENT REPORT
    # =====================================================

    report_df = spark.createDataFrame(
        report_pdf
    )


    (
        report_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .save(
            OUTPUT_PATH
        )
    )


    # =====================================================
    # 17. REPORT SUMMARY
    # =====================================================

    summary = (
        report_df
        .agg(

            F.count(
                "*"
            ).alias(
                "holding_event_count"
            ),

            F.sum(
                "estimated_hold_fuel_kg"
            ).alias(
                "total_estimated_hold_fuel_kg"
            ),

            F.sum(
                "estimated_incremental_fuel_kg"
            ).alias(
                "total_estimated_incremental_fuel_kg"
            ),

            F.sum(
                "estimated_incremental_co2_kg"
            ).alias(
                "total_estimated_incremental_co2_kg"
            ),

            F.avg(
                "estimated_incremental_fuel_kg"
            ).alias(
                "mean_incremental_fuel_kg"
            ),

            F.avg(
                "estimated_incremental_co2_kg"
            ).alias(
                "mean_incremental_co2_kg"
            )
        )

        .withColumn(
            "fuel_model",
            F.lit(
                "OpenAP"
            )
        )

        .withColumn(
            "co2_conversion_factor",
            F.lit(
                CO2_KG_PER_KG_FUEL
            )
        )
    )


    display(
        summary
    )


    # =====================================================
    # 18. EVENT REPORT
    # =====================================================

    display(
        report_df
        .orderBy(
            F.col(
                "estimated_incremental_co2_kg"
            ).desc()
        )
    )

Holding events available: 0
No holding events detected. Fuel/CO2 report created with empty schema.
